# EDA région Auvergne-Rhône-Alpes : stations Citiz, territoire et carte d'accès

**Version sans Parquet.** Sur ce PC, la bibliothèque qui lit les fichiers Parquet (`pyarrow.parquet`) est bloquée par une règle de sécurité. Ce notebook n'en a plus besoin : les sauvegardes se font en CSV, et la motorisation des ménages est calculée seulement si Parquet est disponible (sinon ses colonnes restent vides, le reste de l'analyse ne change pas).

**Questions** :
1. Qu'est-ce qui distingue les communes où Citiz a une station de celles où il n'y en a pas, selon le type de territoire ?
2. L'idée de départ tient-elle : la périphérie des grandes villes, qui travaille au pôle, est-elle desservie ?
3. Où les habitants ont-ils le plus ou le moins d'options de transport à 5 minutes à pied ? (carte d'accès du sujet, pour Power BI)

**Données** (dans `eda-citiz/donnees/`) : stations Citiz (`sorties/stations.csv`), communes (geo.api.gouv.fr), aires d'attraction 2020, flux domicile-travail 2023, dossier complet du recensement 2023 (motorisation), arrêts de transport (GTFS), bornes de recharge (IRVE), population et revenus au carreau de 200 m (Filosofi 2021).

**Limites à garder en tête** :
- On compte des stations, pas des usages : une station présente ne dit pas qu'elle marche. L'usage viendra du relevé de disponibilité Citiz.
- Toutes les distances sont à vol d'oiseau, pas à pied par les rues.

**Ordre** : lancer les cellules dans l'ordre, de haut en bas. Partie 1 (cellules 0 à 10) : on construit les tables. Partie 2 (11 et 12) : on les lit. Partie 3 (14 et 15) : carte d'accès et exports pour Power BI.

## Cellule 0 : imports

Les bibliothèques utilisées dans tout le notebook. Si une importation échoue avec `ModuleNotFoundError`, installer le paquet dans le terminal du `.venv` avec `python -m pip install <paquet>`.

`pyarrow.parquet` est importé dans un `try` : s'il est bloqué, la cellule affiche un message et le notebook continue sans lui (`PARQUET_OK = False`).

In [ ]:
from pathlib import Path
import io, re, time, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import requests
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from shapely.geometry import box
from IPython.display import display

try:
    import pyarrow.parquet as pq
    PARQUET_OK = True
except ImportError as e:
    PARQUET_OK = False
    print("Parquet indisponible (règle de sécurité du PC) : la motorisation des ménages sera laissée vide.")

## Cellule 1 : paramètres et outils

**Paramètres et petites fonctions utilitaires** : chemins, liste des 12 départements d'Auvergne-Rhône-Alpes, projection Lambert 93 (en mètres, nécessaire pour mesurer des distances), lecture des fichiers INSEE, calcul de distances et de comptages autour d'un point. Rien n'est calculé ici ; la dernière ligne affiche les fichiers présents dans `donnees/` pour vérifier que les téléchargements sont au bon endroit.

In [ ]:
EDA = Path(r"C:\Users\bilel\Desktop\epitech\T-HAK-700-MAR-7-1-hack-7\eda-citiz")   # dossier du notebook
DONNEES = EDA / "donnees"                      # fichiers INSEE déjà téléchargés + ceux que les cellules ajoutent
DONNEES.mkdir(parents=True, exist_ok=True)
DEPS_ARA = ["01", "03", "07", "15", "26", "38", "42", "43", "63", "69", "73", "74"]
BBOX_ARA = (2.06, 44.11, 7.19, 46.81)          # lon_min, lat_min, lon_max, lat_max
L93 = "EPSG:2154"                              # Lambert 93, en mètres
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
S = requests.Session()
S.headers["User-Agent"] = "hackathon-mobilite-citiz/0.1"

print(DONNEES.resolve())
print(sorted(p.name for p in DONNEES.iterdir()))

def num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

def telecharger(url, nom):
    """Télécharge une seule fois dans DONNEES, puis réutilise le fichier."""
    chemin = DONNEES / nom
    if not chemin.exists():
        with S.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(chemin, "wb") as f:
                for bloc in r.iter_content(1 << 20):
                    f.write(bloc)
    print(nom, ":", round(chemin.stat().st_size / 1e6, 1), "Mo")
    return chemin

def ressource_datagouv(slug, format_="csv"):
    meta = S.get(f"https://www.data.gouv.fr/api/1/datasets/{slug}/", timeout=30).json()
    res = [r for r in meta["resources"] if (r.get("format") or "").lower() == format_]
    for r in res[:5]:
        print(" -", r["title"], "|", r.get("filesize"), "|", r["url"])
    return res[0]["url"]

def fichier(motif):
    trouves = sorted(DONNEES.glob(motif))
    if not trouves:
        raise FileNotFoundError(f"Aucun fichier {motif} dans {DONNEES}")
    print("Lu :", trouves[-1].name)
    return trouves[-1]

def lire_insee(chemin, cles=("CODGEO",)):
    """Lit un fichier INSEE (csv, xlsx, ou zip qui en contient un) en cherchant la ligne d'en-tête."""
    chemin = Path(chemin)
    if chemin.suffix.lower() == ".zip":
        z = zipfile.ZipFile(chemin)
        noms = [n for n in z.namelist() if n.lower().endswith((".csv", ".xlsx"))]
        nom = max(noms, key=lambda n: z.getinfo(n).file_size)   # le plus gros = les données
        source, ext = io.BytesIO(z.read(nom)), Path(nom).suffix.lower()
    else:
        source, ext = chemin, chemin.suffix.lower()
    if ext == ".csv":
        return pd.read_csv(source, sep=";", dtype=str)
    xls = pd.ExcelFile(source, engine="calamine")   # openpyxl plante sur certains xlsx INSEE (styles invalides)
    for feuille in xls.sheet_names:
        tete = xls.parse(feuille, header=None, nrows=30).astype(str)
        for cle in cles:
            lignes = tete.index[tete.eq(cle).any(axis=1)]
            if len(lignes):
                df = xls.parse(feuille, skiprows=lignes[0], dtype=str)
                return df.rename(columns={cle: "CODGEO"})
    raise ValueError(f"{cles} introuvable dans {chemin.name}, feuilles : {xls.sheet_names}")

def commune_parente(code):
    """Les arrondissements de Lyon, Paris et Marseille sont ramenés à leur commune."""
    code = str(code)
    if "69381" <= code <= "69389": return "69123"
    if "75101" <= code <= "75120": return "75056"
    if "13201" <= code <= "13216": return "13055"
    return code

def sans_arrondissements(df, col="CODGEO"):
    parent = df[col].map(commune_parente)
    doublon = (parent != df[col]) & parent.isin(df[col])   # la commune entière est déjà là
    df = df[~doublon].copy()
    df[col] = df[col].map(commune_parente)
    return df

def compter_autour(xy_de, xy_vers, rayon, poids=None):
    """Nombre (ou somme des poids) de points à moins de `rayon` mètres de chaque point de départ."""
    voisins = cKDTree(xy_vers).query_ball_point(xy_de, rayon)
    if poids is None:
        return np.array([len(v) for v in voisins])
    poids = np.asarray(poids, dtype=float)
    return np.array([poids[v].sum() for v in voisins])

def plus_proche(xy_de, xy_vers):
    return cKDTree(xy_vers).query(xy_de)[0]

## Cellule 2 : communes de la région (contours, population)

**Les communes de la région** (contours, population, surface), téléchargées une fois par département depuis geo.api.gouv.fr puis gardées dans `donnees/`. C'est l'unité d'analyse : on regarde quelles communes ont une station et lesquelles n'en ont pas. *À vérifier : environ 4 000 communes et 8 millions d'habitants.*

In [ ]:
def communes_dep(dep):
    f = DONNEES / f"communes_{dep}.geojson"
    if not f.exists():
        url = (f"https://geo.api.gouv.fr/departements/{dep}/communes"
               "?fields=nom,code,population,surface,codeDepartement&format=geojson&geometry=contour")
        for essai in range(5):
            try:
                r = S.get(url, timeout=(15, 300))
                r.raise_for_status()
                f.write_bytes(r.content)
                break
            except requests.RequestException as e:
                print(dep, "essai", essai + 1, "échec :", type(e).__name__)
                time.sleep(5 * (essai + 1))
        else:
            raise RuntimeError(f"Département {dep} : l'API ne répond pas, relance la cellule plus tard")
    print(dep, "ok")
    return gpd.read_file(f)

cache = DONNEES / "communes_ara.gpkg"
if cache.exists():
    com = gpd.read_file(cache)
else:
    com = pd.concat([communes_dep(d) for d in DEPS_ARA], ignore_index=True)
    com = com.set_crs("EPSG:4326", allow_override=True).to_crs(L93)
    com.to_file(cache)
com["population"] = com["population"].fillna(0)
com["surface_km2"] = com["surface"] / 100            # l'API donne la surface en hectares
com["densite_hab_km2"] = com["population"] / com["surface_km2"]
print(len(com), "communes,", int(com["population"].sum()), "habitants")
com.head(3)

## Cellule 3 : stations Citiz, rattachées aux communes

**Les 622 stations Citiz réelles** (sans les deux fausses stations), rattachées à leur commune par position. On compte ensuite les stations par commune et on marque les communes « desservies ». *À vérifier : 622 stations ; le nombre de communes desservies doit être proche de 130.*

In [ ]:
st = pd.read_csv(EDA / "sorties" / "stations.csv", dtype={"code_insee": str, "departement": str})
st = st[(st["region"] == "Auvergne-Rhône-Alpes") & (~st["ouverture_prochaine"].astype(bool))]
st = st[~st["nom"].str.contains("hors-circuit|DEMOBOX", case=False)].copy()   # les 2 fausses stations
st = gpd.GeoDataFrame(st, geometry=gpd.points_from_xy(st["lon"], st["lat"]), crs="EPSG:4326").to_crs(L93)
st = gpd.sjoin(st, com[["code", "geometry"]], how="left", predicate="within").drop(columns="index_right")
print(len(st), "stations (attendu : 622)")
print("Commune différente de la première EDA :", int((st["code"] != st["code_insee"]).sum()))
XY_ST = np.c_[st.geometry.x, st.geometry.y]

com = com.merge(st.groupby("code").size().rename("nb_stations"), left_on="code", right_index=True, how="left")
com["nb_stations"] = com["nb_stations"].fillna(0).astype(int)
com["desservie"] = com["nb_stations"] > 0
print(int(com["desservie"].sum()), "communes desservies sur", len(com))

## Cellule 4 : type de territoire (aires d'attraction et densité)

**Type de territoire** : chaque commune est classée selon son aire d'attraction des villes (INSEE) : taille de l'aire (grande, moyenne, petite, hors aire) et position (pôle ou couronne). C'est la grille qui permet de tester l'idée « périphérie des grandes villes ». Le tableau final compte, par type de territoire, les communes, les habitants, les communes desservies et les stations.

In [ ]:
aav = lire_insee(fichier("AAV2020*"))
dens = lire_insee(fichier("fichier_diffusion*"), cles=("CODGEO", "Code commune", "COM"))
print("AAV :", aav.columns.tolist())
print("Densité :", dens.columns.tolist())

col_dens = [c for c in dens.columns if "DENS" in c.upper() and "LIB" not in c.upper()][0]
col_libdens = [c for c in dens.columns if "LIBDENS" in c.upper() or "LIBELLÉ" in c.upper()]
dens = dens[["CODGEO", col_dens] + col_libdens[:1]].rename(columns={col_dens: "dens"})
if col_libdens:
    dens = dens.rename(columns={col_libdens[0]: "lib_dens"})
print("Colonne densité retenue :", col_dens)

com = com.merge(aav[["CODGEO", "AAV2020", "LIBAAV2020", "CATEAAV2020"]], left_on="code", right_on="CODGEO", how="left").drop(columns="CODGEO")
com = com.merge(dens, left_on="code", right_on="CODGEO", how="left").drop(columns="CODGEO")

# Position dans l'aire : 11 commune-centre, 12 autre commune du pôle principal, 13 pôle secondaire, 20 couronne, 30 hors attraction
POSITION = {"11": "pôle", "12": "pôle", "13": "pôle secondaire", "20": "couronne", "30": "hors aire"}
com["position"] = com["CATEAAV2020"].astype(str).str[:2].map(POSITION)
pop_aire = com[com["position"] != "hors aire"].groupby("AAV2020")["population"].sum()   # partie située dans la région
com["pop_aire"] = com["AAV2020"].map(pop_aire)
com["taille_aire"] = pd.cut(com["pop_aire"], [0, 100_000, 400_000, np.inf],
                            labels=["3 petite aire (< 100 000)", "2 aire moyenne (100 à 400 000)", "1 grande aire (≥ 400 000)"]).astype(str)
com.loc[com["position"] == "hors aire", "taille_aire"] = "4 hors aire"
com["territoire"] = com["taille_aire"].str.replace(r" \(.*\)", "", regex=True) + " · " + com["position"]

print("Communes sans aire trouvée :", int(com["position"].isna().sum()))
display(com.groupby("territoire").agg(communes=("code", "size"), habitants=("population", "sum"),
                                      desservies=("desservie", "sum"), stations=("nb_stations", "sum")))
display(pop_aire.sort_values(ascending=False).head(10).rename(lambda a: com.loc[com["AAV2020"] == a, "LIBAAV2020"].iloc[0]))

## Cellule 5 : flux domicile-travail

**Flux domicile-travail** (recensement 2023) : pour chaque commune, part des actifs qui travaillent hors de leur commune, au pôle de leur aire, ou dans un grand pôle. Les arrondissements de Lyon, Paris et Marseille sont regroupés. *Contrôle : les actifs doivent représenter environ 40 à 45 % des habitants.*

In [ ]:
flux = lire_insee(fichier("base-flux-mobilite-domicile-lieu-travail*"))
print(flux.columns.tolist(), len(flux), "lignes")
col_n = next(c for c in flux.columns if c.upper().startswith("NBFLUX"))
flux["n"] = num(flux[col_n])
flux["orig"] = flux["CODGEO"].map(commune_parente)
flux["dest"] = flux["DCLT"].map(commune_parente)
flux = flux.groupby(["orig", "dest"], as_index=False)["n"].sum()

info = com.set_index("code")[["AAV2020", "position", "taille_aire"]]
flux = flux.join(info.add_suffix("_o"), on="orig").join(info.add_suffix("_d"), on="dest")
ailleurs = flux["orig"] != flux["dest"]
vers_pole_aire = ailleurs & (flux["position_d"] == "pôle") & (flux["AAV2020_o"] == flux["AAV2020_d"])
vers_grand_pole = ailleurs & (flux["position_d"] == "pôle") & flux["taille_aire_d"].astype(str).str.startswith("1")

f = pd.DataFrame({
    "actifs": flux.groupby("orig")["n"].sum(),
    "emplois": flux.groupby("dest")["n"].sum(),
    "sur_place": flux[~ailleurs].groupby("orig")["n"].sum(),
    "vers_pole_aire": flux[vers_pole_aire].groupby("orig")["n"].sum(),
    "vers_grand_pole": flux[vers_grand_pole].groupby("orig")["n"].sum(),
}).fillna(0)
f["part_sortants"] = 1 - f["sur_place"] / f["actifs"]
f["part_vers_pole_aire"] = f["vers_pole_aire"] / f["actifs"]
f["part_vers_grand_pole"] = f["vers_grand_pole"] / f["actifs"]
f["emplois_par_actif"] = f["emplois"] / f["actifs"]
com = com.merge(f, left_on="code", right_index=True, how="left")

# Contrôle : les actifs occupés font environ 40 à 45 % de la population. Beaucoup moins = petits flux coupés par l'INSEE.
print("Actifs / habitants dans la région :", round(com["actifs"].sum() / com["population"].sum(), 3))
print("Communes sans aucun flux :", int(com["actifs"].isna().sum()))
display(com.groupby("territoire")[["part_sortants", "part_vers_pole_aire", "part_vers_grand_pole", "emplois_par_actif"]].median().round(2))

## Cellule 5b : extraire les communes du dossier complet (une seule fois, seulement si Parquet est disponible)

Le « dossier complet » de l'INSEE est un très gros fichier Parquet au format long (une ligne par commune, tableau, mesure et année). On en extrait les communes pour quelques tableaux (voitures des ménages, résidences principales, mode de transport, revenus) et on enregistre un petit fichier **CSV**, `dossier_communes.csv`, que la cellule suivante lit en une seconde.

- Si `dossier_communes.csv` existe déjà : la cellule ne fait rien.
- Si Parquet est bloqué et que le CSV n'existe pas : la cellule le dit et passe. La motorisation sera vide.

In [ ]:
TABLEAUX = ["LOG_T12", "LOG_T10", "LOG_T5", "FAM_T1", "ACT_G2", "REV_T1"]
cols = ["GEO", "GEO_OBJECT", "TIME_PERIOD", "ID_TAB", "TAB_MEASURE", "TAB_MEASURE_LABEL", "OBS_VALUE"]
dossier_csv = DONNEES / "dossier_communes.csv"
if dossier_csv.exists():
    print("Déjà extrait :", dossier_csv.name)
elif not PARQUET_OK:
    print("Parquet bloqué et", dossier_csv.name, "absent : étape ignorée, la motorisation restera vide.")
else:
    pf = pq.ParquetFile(DONNEES / "dossier_complet.parquet")
    morceaux = []
    for lot in pf.iter_batches(batch_size=500_000, columns=cols):
        d = lot.to_pandas()
        morceaux.append(d[d["ID_TAB"].isin(TABLEAUX) & (d["GEO_OBJECT"] == "COM")])
    dossier = pd.concat(morceaux, ignore_index=True)
    dossier.to_csv(dossier_csv, index=False)
    print(len(dossier), "lignes enregistrées dans", dossier_csv.name)

## Cellule 6 : motorisation des ménages

**Motorisation des ménages** (recensement 2023) : part des ménages sans voiture et part des ménages avec deux voitures ou plus, par commune. C'est la variable clé de l'autopartage : plus la part sans voiture est élevée, plus un partage de voiture a de sens.

Si `dossier_communes.csv` n'existe pas (Parquet bloqué), les trois colonnes sont créées **vides** pour que la suite du notebook fonctionne. Les cartes d'accès (Partie 3) ne les utilisent pas.

In [ ]:
dossier_csv = DONNEES / "dossier_communes.csv"
if dossier_csv.exists():
    dossier = pd.read_csv(dossier_csv, dtype={"GEO": str, "TIME_PERIOD": str, "ID_TAB": str, "TAB_MEASURE": str})
    d23 = dossier[dossier["TIME_PERIOD"].astype(str) == "2023"]
    voit = (d23[d23["ID_TAB"] == "LOG_T12"]
            .pivot_table(index="GEO", columns="TAB_MEASURE", values="OBS_VALUE", aggfunc="sum"))
    menages = (d23[(d23["ID_TAB"] == "LOG_T5") & (d23["TAB_MEASURE"] == "DWELLINGS_OCS_DW_MAIN")]
               .groupby("GEO")["OBS_VALUE"].sum())
    moto = pd.DataFrame({"menages": menages,
                         "avec_voiture": voit["DWELLINGS_OCS_DW_MAIN_CARS_C_GE1"],
                         "deux_voitures": voit["DWELLINGS_OCS_DW_MAIN_CARS_C_GE2"]})
    moto["part_sans_voiture"] = (1 - moto["avec_voiture"] / moto["menages"]).clip(0, 1)
    moto["part_2_voitures"] = moto["deux_voitures"] / moto["menages"]
    print(len(moto), "communes ; ménages en France :", int(moto["menages"].sum()))
    com = com.merge(moto[["menages", "part_sans_voiture", "part_2_voitures"]], left_on="code", right_index=True, how="left")
    print("Communes sans motorisation :", int(com["part_sans_voiture"].isna().sum()))
    display(com.groupby("territoire")[["part_sans_voiture", "part_2_voitures"]].median().round(3))
else:
    for c in ["menages", "part_sans_voiture", "part_2_voitures"]:
        com[c] = np.nan
    print("Motorisation non calculée (pas de dossier_communes.csv) : colonnes laissées vides.")

## Cellule 7 : arrêts de transport en commun

**Arrêts de transport en commun** (agrégat national GTFS), filtrés sur la région et dédoublonnés. Pour chaque station : nombre d'arrêts à moins de 300 m et distance à l'arrêt le plus proche ; pour chaque commune : arrêts pour 1 000 habitants. On ne connaît pas encore la fréquence de passage, seulement la présence des arrêts.

In [ ]:
url = ressource_datagouv("arrets-de-transport-en-france")
arrets = pd.read_csv(telecharger(url, "arrets_france.csv"), low_memory=False)
print(arrets.columns.tolist())
x0, y0, x1, y1 = BBOX_ARA
arrets = arrets[arrets["stop_lon"].between(x0, x1) & arrets["stop_lat"].between(y0, y1)]
if "location_type" in arrets:
    arrets = arrets[arrets["location_type"].fillna(0).astype(str).isin(["0", "0.0"])]   # quais, pas les zones d'arrêt
# Dédoublonnage : même nom à moins d'environ 10 m (plusieurs réseaux publient le même arrêt)
arrets["cle"] = (arrets["stop_name"].astype(str).str.lower().str.strip() + "|"
                 + arrets["stop_lat"].round(4).astype(str) + "|" + arrets["stop_lon"].round(4).astype(str))
avant = len(arrets)
arrets = arrets.drop_duplicates("cle")
arrets = gpd.GeoDataFrame(arrets, geometry=gpd.points_from_xy(arrets["stop_lon"], arrets["stop_lat"]), crs="EPSG:4326").to_crs(L93)
arrets = gpd.sjoin(arrets, com[["code", "geometry"]], how="inner", predicate="within")
print(avant, "arrêts dans le cadre,", len(arrets), "après dédoublonnage et dans la région")
XY_ARRETS = np.c_[arrets.geometry.x, arrets.geometry.y]

st["arrets_300m"] = compter_autour(XY_ST, XY_ARRETS, 300)
st["dist_arret_m"] = plus_proche(XY_ST, XY_ARRETS)
com = com.merge(arrets.groupby("code").size().rename("nb_arrets"), left_on="code", right_index=True, how="left")
com["nb_arrets"] = com["nb_arrets"].fillna(0)
com["arrets_pour_1000_hab"] = com["nb_arrets"] / com["population"].clip(lower=1) * 1000
print(st["dist_arret_m"].describe().round(0))

## Cellule 8 : bornes de recharge publiques

**Bornes de recharge publiques** (fichier consolidé IRVE) : nombre de points de charge à moins de 300 m de chaque station et distance à la plus proche. Cela prépare la question « la voiture peut-elle être électrique ? ». La puissance est en kW (médiane autour de 22).

In [ ]:
url = ressource_datagouv("fichier-consolide-des-bornes-de-recharge-pour-vehicules-electriques")
irve = pd.read_csv(telecharger(url, "irve_consolide.csv"), low_memory=False)
print(irve.columns.tolist()[:40])
col_lon = next(c for c in irve.columns if "longitude" in c.lower())
col_lat = next(c for c in irve.columns if "latitude" in c.lower())
col_kw = next(c for c in irve.columns if "puissance_nominale" in c.lower())
irve["lon"], irve["lat"], irve["kw"] = num(irve[col_lon]), num(irve[col_lat]), num(irve[col_kw])
x0, y0, x1, y1 = BBOX_ARA
irve = irve[irve["lon"].between(x0, x1) & irve["lat"].between(y0, y1)]
irve = gpd.GeoDataFrame(irve, geometry=gpd.points_from_xy(irve["lon"], irve["lat"]), crs="EPSG:4326").to_crs(L93)
irve = gpd.sjoin(irve, com[["code", "geometry"]], how="inner", predicate="within")
print(len(irve), "points de charge dans la région")
print("Puissance (kW) :", irve["kw"].describe().round(1).to_dict())   # certains fichiers donnent des W : à vérifier
XY_IRVE = np.c_[irve.geometry.x, irve.geometry.y]

st["irve_300m"] = compter_autour(XY_ST, XY_IRVE, 300)
st["dist_irve_m"] = plus_proche(XY_ST, XY_IRVE)
com = com.merge(irve.groupby("code").size().rename("nb_points_charge"), left_on="code", right_index=True, how="left")
com["nb_points_charge"] = com["nb_points_charge"].fillna(0)
print("Stations avec un point de charge à moins de 300 m :", int((st["irve_300m"] > 0).sum()), "sur", len(st))

## Cellule 9 : population et revenus au carreau de 200 m

**Population et revenus au carreau de 200 m** (Filosofi 2021, France métropolitaine) : habitants, ménages, niveau de vie et part de ménages pauvres dans un rayon de 500 m autour de chaque station. On ne lit que le cadre de la région. *Contrôle : le total d'habitants doit être proche de celui de la cellule 2.*

In [ ]:
cadre = gpd.GeoSeries([box(*BBOX_ARA)], crs="EPSG:4326")
gpkg = DONNEES / "filosofi" / "carreaux_200m_met.gpkg"        # France métropolitaine (les autres : mart = Martinique, reun = La Réunion)
couches = pyogrio.list_layers(gpkg)
print("Fichier :", gpkg.name, "| couches :", couches.tolist())
couche = couches[0][0]
print("Couche lue :", couche)
car = gpd.read_file(gpkg, layer=couche, bbox=cadre)     # ne lit que le cadre de la région
car.columns = [c.lower() if c != "geometry" else c for c in car.columns]
print(len(car), "carreaux lus ;", car.columns.tolist())
car = car.to_crs(L93)
car["geometry"] = car.geometry.centroid
car = gpd.sjoin(car, com[["code", "geometry"]], how="inner", predicate="within").drop(columns="index_right")
XY_CAR = np.c_[car.geometry.x, car.geometry.y]
print(len(car), "carreaux dans la région,", int(car["ind"].sum()), "habitants (à comparer à la cellule 2)")

st["pop_500m"] = compter_autour(XY_ST, XY_CAR, 500, car["ind"])
st["men_500m"] = compter_autour(XY_ST, XY_CAR, 500, car["men"])
st["niveau_vie_500m"] = compter_autour(XY_ST, XY_CAR, 500, car["ind_snv"]) / st["pop_500m"].clip(lower=1)
st["part_pauvres_500m"] = compter_autour(XY_ST, XY_CAR, 500, car["men_pauv"]) / st["men_500m"].clip(lower=1)

rev = car.groupby("code")[["ind", "ind_snv"]].sum()
com = com.merge((rev["ind_snv"] / rev["ind"]).rename("niveau_vie_moyen"), left_on="code", right_index=True, how="left")
car["dist_station_m"] = plus_proche(XY_CAR, XY_ST)        # servira à la partie 2
print(st[["pop_500m", "niveau_vie_500m", "part_pauvres_500m"]].describe().round(2))

## Cellule 10 : assembler et enregistrer

**Assemblage et sauvegarde** : une table par station, une par commune et une par carreau, enregistrées en **CSV** dans `donnees/` (`stations_croisees.csv`, `communes_croisees.csv`, `carreaux_ara.csv`) pour ne pas tout recalculer. Le tableau final donne les valeurs médianes par type de territoire ; la colonne « part_sans_voiture » sera vide si la motorisation n'a pas pu être calculée.

In [ ]:
VARS_COM = ["code", "territoire", "taille_aire", "position", "LIBAAV2020", "dens", "densite_hab_km2", "population",
            "part_sortants", "part_vers_pole_aire", "part_vers_grand_pole", "emplois_par_actif",
            "part_sans_voiture", "part_2_voitures", "arrets_pour_1000_hab", "niveau_vie_moyen"]
st_x = st.drop(columns=["population"], errors="ignore").merge(com[VARS_COM], on="code", how="left")
st_x.drop(columns="geometry").to_csv(DONNEES / "stations_croisees.csv", index=False)
com.drop(columns="geometry").to_csv(DONNEES / "communes_croisees.csv", index=False)
car.drop(columns="geometry").assign(x=XY_CAR[:, 0], y=XY_CAR[:, 1]).to_csv(DONNEES / "carreaux_ara.csv", index=False)

print("Valeurs manquantes par variable (stations) :")
print(st_x[["territoire", "part_vers_grand_pole", "part_sans_voiture", "pop_500m", "dist_arret_m", "dist_irve_m"]].isna().sum())
display(st_x.groupby("territoire")[["pop_500m", "niveau_vie_500m", "arrets_300m", "irve_300m", "part_sans_voiture"]].median().round(2))

# Partie 2 : zones desservies et zones vides

À lancer quand la partie 1 passe. On compare maintenant les zones desservies et les zones vides.

## Cellule 11 : qui est couvert, par type de territoire

**Qui est couvert ?** Par type de territoire : part des communes desservies, nombre de stations pour 100 000 habitants, part des habitants à moins de 500 m d'une station. Puis comparaison des communes avec et sans station (médianes) : population, densité, flux vers le pôle, motorisation, arrêts, niveau de vie, bornes.

In [ ]:
car["a_500m"] = car["dist_station_m"] <= 500
car = car.merge(com[["code", "territoire"]], on="code", how="left")
print(len(car), "carreaux,", int(car["territoire"].isna().sum()), "sans territoire,", int(car["a_500m"].sum()), "à moins de 500 m d'une station")
car["hab_a_500m"] = car["ind"] * car["a_500m"]
couv = car.groupby("territoire").agg(habitants=("ind", "sum"), hab_a_500m=("hab_a_500m", "sum"))
couv["part_hab_a_500m_station"] = couv["hab_a_500m"] / couv["habitants"]
desserte = com.groupby("territoire").agg(communes=("code", "size"), part_communes_desservies=("desservie", "mean"),
                                         stations=("nb_stations", "sum"), habitants=("population", "sum"))
desserte["stations_pour_100k_hab"] = desserte["stations"] / desserte["habitants"] * 1e5
display(desserte.join(couv["part_hab_a_500m_station"]).round(3))

VARS = ["population", "densite_hab_km2", "part_vers_grand_pole", "part_vers_pole_aire", "part_sortants",
        "part_sans_voiture", "arrets_pour_1000_hab", "niveau_vie_moyen", "nb_points_charge"]
display(com.groupby(["territoire", "desservie"])[VARS].median().round(2))

## Cellule 12 : couronne et petites communes, selon les flux vers le pôle

**Test de l'idée de départ** : dans la couronne des grandes villes et dans les petites communes, la présence de Citiz dépend-elle de la taille de la commune et de la part des actifs qui vont travailler au pôle ? Le tableau croise les deux, le nuage de points montre les communes avec et sans station, et la liste donne des communes de profil comparable sans station (premières pistes, pas encore un classement).

In [ ]:
zone = com[com["position"].isin(["couronne", "pôle secondaire", "hors aire"])].copy()
zone["taille_commune"] = pd.cut(zone["population"], [0, 2_000, 5_000, 10_000, np.inf],
                                labels=["< 2 000", "2 à 5 000", "5 à 10 000", "≥ 10 000"])
zone["flux_vers_pole"] = pd.cut(zone["part_vers_pole_aire"], [-0.01, 0.2, 0.4, 1],
                                labels=["< 20 %", "20 à 40 %", "> 40 %"])
display(zone.pivot_table(index="taille_commune", columns="flux_vers_pole", values="desservie",
                         aggfunc=["mean", "size"], observed=False).round(2))

fig, ax = plt.subplots(figsize=(8, 5))
for d, coul in [(False, "lightgray"), (True, "tab:red")]:
    z = zone[zone["desservie"] == d]
    ax.scatter(z["population"], z["part_vers_pole_aire"], s=8 if not d else 20, c=coul,
               label="avec station" if d else "sans station", alpha=0.6)
ax.set_xscale("log"); ax.set_xlabel("Habitants (échelle log)"); ax.set_ylabel("Part des actifs qui travaillent au pôle de l'aire")
ax.set_title("Communes hors pôle : présence de Citiz selon la taille et les flux vers le pôle")
ax.legend(); plt.tight_layout(); plt.show()

# Communes qui ressemblent aux communes desservies mais n'ont pas de station : premières pistes, pas encore un classement
pistes = zone[(~zone["desservie"]) & (zone["population"] >= 2_000) & (zone["part_vers_pole_aire"] >= 0.3) & (zone["nb_arrets"] > 0)]
display(pistes.sort_values("population", ascending=False)[["nom", "LIBAAV2020", "territoire", "population", "part_vers_pole_aire",
                                                             "part_sans_voiture", "nb_arrets", "nb_points_charge"]].head(25))
print(len(pistes), "communes de ce profil sans station")

# Partie 3 : carte d'accès (objectif 1 du sujet)

Le sujet demande une carte où chaque zone est colorée **du froid au chaud** selon le **nombre d'options de transport accessibles par habitant à 5 minutes à pied**. Les deux cellules suivantes calculent cette mesure sur toute la région et exportent des fichiers pour Power BI.

(La cellule 13, test du réseau piéton OpenStreetMap, est volontairement absente : les serveurs publics étaient saturés. Elle est décrite dans `cellule_13_reseau_pieton.md`.)

## Cellule 14 : calcul de l'accès à 5 minutes à pied et export 1 km

**Ce que fait la cellule**
- 5 minutes à pied = 400 m de marche. Sans réseau piéton, on prend 400 / 1,3 ≈ **308 m à vol d'oiseau** (1,3 = coefficient de détour).
- **Options par habitant** : chaque arrêt de transport en commun et chaque station Citiz est **partagé entre tous les habitants qui l'atteignent**. Un carreau de 5 habitants à côté d'un arrêt n'apparaît donc pas « sur-desservi ».
- Secret statistique : les carreaux de moins de 11 habitants sont exclus des exports.
- **Deux indicateurs** :
  - `options_pour_1000_hab` (celui du sujet) : mesure par habitant. Attention, elle **favorise les zones peu denses** : un village avec un arrêt pour 100 habitants score plus haut que le centre de Lyon avec 15 arrêts pour 3 000 habitants.
  - `options_5min` : nombre brut d'options à 5 min (arrêts + stations Citiz), moyenné sur les habitants du carré. Il se lit comme on l'attend : ville dense = beaucoup, campagne = peu.
- Niveaux : pour l'indicateur par habitant, « 0 aucune option » puis 4 quartiles des valeurs positives (`niveau`). Pour le nombre brut, des seuils fixes : 0, 1 à 2, 3 à 5, 6 à 10, plus de 10 (`niveau_nb`).
- Export dans `donnees/powerbi/` : `carte_acces_1km.csv` (un point par carré de 1 km, moins de 30 000 points, la limite d'un visuel Carte, avec les deux indicateurs et leurs niveaux), `acces_par_commune.csv`. Copie des arrêts : `donnees/arrets_ara.csv`.

**Temps** : quelques minutes. **À vérifier** : environ 2,88 M d'habitants sans option sur 6,92 M (42 %), environ 920 000 habitants avec une station Citiz à 5 min (13 %), 23 244 carrés de 1 km exportés.

**Limites** : distance à vol d'oiseau, seulement arrêts et stations Citiz (ni vélos, ni trottinettes, ni autre autopartage, ni fréquence de passage), un arrêt compte une fois quel que soit le nombre de lignes.

In [ ]:
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

R = 400 / 1.3                      # 5 min à pied, en mètres à vol d'oiseau
ind = car["ind"].to_numpy(dtype=float)
arbre_car = cKDTree(XY_CAR)

def options_par_habitant(xy_points):
    """Pour chaque carreau : somme des parts de chaque point atteint (1 / habitants qui l'atteignent)."""
    voisins = arbre_car.query_ball_point(xy_points, R)
    demande = np.array([ind[v].sum() if len(v) else 0.0 for v in voisins])
    part = np.where(demande > 0, 1.0 / np.maximum(demande, 1), 0.0)
    atteints = cKDTree(xy_points).query_ball_point(XY_CAR, R)
    nb = np.array([len(a) for a in atteints])
    score = np.array([part[a].sum() if len(a) else 0.0 for a in atteints])
    return nb, score

nb_tc, sc_tc = options_par_habitant(XY_ARRETS)
nb_ct, sc_ct = options_par_habitant(XY_ST)

carte = pd.DataFrame({"x": XY_CAR[:, 0], "y": XY_CAR[:, 1], "habitants": ind,
                      "nb_arrets_5min": nb_tc, "nb_stations_citiz_5min": nb_ct,
                      "options_pour_1000_hab": 1000 * (sc_tc + sc_ct),
                      "tc_pour_1000_hab": 1000 * sc_tc, "citiz_pour_1000_hab": 1000 * sc_ct})
carte["code"] = car["code"].to_numpy()

vus = carte[carte["habitants"] >= 11].copy()        # secret statistique : pas de carreau de moins de 11 habitants
vus["nb_options_5min"] = vus.nb_arrets_5min + vus.nb_stations_citiz_5min   # nombre brut d'options à 5 min (arrêts + stations Citiz)
sans_option = vus["nb_options_5min"] == 0

def niveau_nb(v):
    """Niveaux du nombre brut d'options : seuils fixes, faciles à lire."""
    n = pd.cut(v, [-0.01, 0, 2, 5, 10, np.inf], labels=["0 aucune option", "1 faible (1 à 2)", "2 moyen (3 à 5)", "3 élevé (6 à 10)", "4 très élevé (plus de 10)"])
    return n.astype(str)
print("Habitants sans aucune option à 5 min :", int(vus.loc[sans_option, "habitants"].sum()), "sur", int(vus["habitants"].sum()))
print("Habitants avec une station Citiz à 5 min :", int(vus.loc[vus.nb_stations_citiz_5min > 0, "habitants"].sum()))

# Carrés de 1 km (moins de 30 000 points : la limite d'un visuel Carte)
vus["kx"] = vus.x // 1000
vus["ky"] = vus.y // 1000
vus["opt_x_hab"] = vus.options_pour_1000_hab * vus.habitants
vus["nb_x_hab"] = vus.nb_options_5min * vus.habitants
agg = vus.groupby(["kx", "ky"]).agg(habitants=("habitants", "sum"), opt_x_hab=("opt_x_hab", "sum"),
                                    nb_x_hab=("nb_x_hab", "sum")).reset_index()
agg["options_pour_1000_hab"] = agg["opt_x_hab"] / agg["habitants"]
agg["options_5min"] = agg["nb_x_hab"] / agg["habitants"]        # nombre moyen d'options à 5 min pour un habitant du carré

# Aperçu froid / chaud
fig, ax = plt.subplots(figsize=(9, 9))
sc = ax.scatter(agg.kx * 1000, agg.ky * 1000, c=agg.options_pour_1000_hab, s=3, cmap="coolwarm",
                vmax=agg.options_pour_1000_hab.quantile(0.95))
plt.colorbar(sc, label="options de transport pour 1 000 habitants (5 min à pied)")
ax.set_aspect("equal"); ax.set_title("Carte d'accès, Auvergne-Rhône-Alpes (carrés de 1 km)")
plt.show()

# Export pour Power BI / Fabric
OUT = DONNEES / "powerbi"
OUT.mkdir(exist_ok=True)

pts = gpd.GeoSeries(gpd.points_from_xy(agg.kx * 1000 + 500, agg.ky * 1000 + 500), crs=L93).to_crs(4326)
sortie = agg.assign(lon=pts.x.round(5), lat=pts.y.round(5))[["lon", "lat", "habitants", "options_pour_1000_hab", "options_5min"]]
sortie["options_5min"] = sortie["options_5min"].round(2)
sortie["niveau_nb"] = niveau_nb(sortie["options_5min"])
positif = sortie["options_pour_1000_hab"] > 0       # plus de la moitié des valeurs sont 0 : niveaux sur les valeurs positives
sortie["niveau"] = "0 aucune option"
sortie.loc[positif, "niveau"] = pd.qcut(sortie.loc[positif, "options_pour_1000_hab"], 4,
                                        labels=["1 faible", "2 moyen", "3 élevé", "4 très élevé"]).astype(str)
sortie.to_csv(OUT / "carte_acces_1km.csv", index=False, encoding="utf-8-sig")

vus["hab_sans"] = vus.habitants.where(sans_option, 0)
par_commune = vus.groupby("code").agg(habitants=("habitants", "sum"), opt_x_hab=("opt_x_hab", "sum"),
                                      hab_sans=("hab_sans", "sum")).reset_index()
par_commune["options_pour_1000_hab"] = par_commune["opt_x_hab"] / par_commune["habitants"]
par_commune["part_hab_sans_option"] = par_commune["hab_sans"] / par_commune["habitants"]
par_commune[["code", "habitants", "options_pour_1000_hab", "part_hab_sans_option"]].to_csv(
    OUT / "acces_par_commune.csv", index=False, encoding="utf-8-sig")

pd.DataFrame(XY_ARRETS, columns=["x", "y"]).to_csv(DONNEES / "arrets_ara.csv", index=False)   # CSV : pyarrow est bloqué sur ce PC
print(len(sortie), "carrés exportés dans", OUT)
print(sortie["niveau"].value_counts())
print(sortie["niveau_nb"].value_counts())

## Cellule 15 : export des carreaux de 200 m (page de détail Power BI)

**Ce que fait la cellule** : exporte `donnees/powerbi/carte_acces_200m.csv` : un point par carreau de 200 m (au moins 11 habitants), avec la commune, l'aire d'attraction, le type de territoire, les habitants, les options pour 1 000 habitants, le nombre d'arrêts et de stations Citiz à 5 min, le nombre brut d'options (`nb_options_5min`) et les deux niveaux (`niveau`, `niveau_nb`). Ce fichier sert à la page « détail » de Power BI : zoom sur une ville, filtres par aire d'attraction, commune et type de territoire.

À lancer après la cellule 14 (elle réutilise `vus`). À vérifier : le nombre de carreaux exportés ; un visuel Carte de Power BI n'affiche pas plus d'environ 30 000 points à la fois, donc on filtre par aire ou par commune.

In [ ]:
nom_commune = com.set_index("code")["nom"]
aire = com.set_index("code")["LIBAAV2020"]

fin = vus[["x", "y", "habitants", "options_pour_1000_hab", "nb_arrets_5min", "nb_stations_citiz_5min", "code"]].copy()
pts = gpd.GeoSeries(gpd.points_from_xy(fin.x, fin.y), crs=L93).to_crs(4326)
fin["lon"] = pts.x.round(5).to_numpy()
fin["lat"] = pts.y.round(5).to_numpy()
fin["commune"] = fin["code"].map(nom_commune)
fin["aire"] = fin["code"].map(aire).fillna("Hors aire d'attraction")
fin["territoire"] = fin["code"].map(com.set_index("code")["territoire"])
fin["nb_options_5min"] = fin["nb_arrets_5min"] + fin["nb_stations_citiz_5min"]
fin["niveau_nb"] = niveau_nb(fin["nb_options_5min"])
fin["habitants"] = fin["habitants"].round(0).astype(int)
fin["options_pour_1000_hab"] = fin["options_pour_1000_hab"].round(2)

positif = fin["options_pour_1000_hab"] > 0
fin["niveau"] = "0 aucune option"
fin.loc[positif, "niveau"] = pd.qcut(fin.loc[positif, "options_pour_1000_hab"], 4,
                                     labels=["1 faible", "2 moyen", "3 élevé", "4 très élevé"]).astype(str)

cols = ["lon", "lat", "commune", "aire", "territoire", "habitants", "options_pour_1000_hab",
        "nb_arrets_5min", "nb_stations_citiz_5min", "nb_options_5min", "niveau", "niveau_nb"]
fin[cols].to_csv(OUT / "carte_acces_200m.csv", index=False, encoding="utf-8-sig")
print(len(fin), "carreaux de 200 m exportés")
print(fin.groupby("aire").size().sort_values(ascending=False).head(10))

## Ce qu'on retiendra (à compléter après exécution)

- Où sont les stations, par type de territoire (cellules 4 et 11).
- Les communes de la couronne des grandes villes avec station ont-elles plus d'actifs qui vont travailler au pôle ? (cellule 12)
- Les petites communes sans station : combien ressemblent aux communes desservies ? (liste de la cellule 12)
- Part des habitants sans aucune option à 5 min et part avec une station Citiz (cellule 14), avec la limite : distances à vol d'oiseau.
- Ce que l'EDA ne dit pas : l'usage réel des stations. Prochaine étape : relier ces variables à l'occupation observée dans le relevé de disponibilité.